# Algorithms 19: SIR Model For the Spread of Disease**Learning Objectives:**1. Formulate the SIR (Susceptible, Infected, Recovered) ODE equations2. Understand population dynamics and transmission rates3. Simulate an epidemic outbreak using Multivariable Euler Estimation**Prerequisites:** Algorithms 18**Estimated time:** 60 minutes**Textbook:** *Justin Skycak — Intro to Algorithms & Machine Learning* Chapter 19

In [ ]:
# Google Colab Setupimport mathimport matplotlib.pyplot as pltprint('Ready ✅')

---## Phase −1 — Theoretical Foundation### The SIR Compartmental Model> 📖 *From Algorithms Ch. 19:* The SIR model assumes three sub-populations: Susceptible ($S$), Infected ($I$), and Recovered ($R$).Let's model the rate of change of each group over time $t$ (days):**1. Susceptible Population ($S$)**Susceptible people decrease as they catch the disease from infected people. The number of meetings between the groups is proportional to $S \times I$.$\frac{dS}{dt} = - \text{transmission\_rate} \times \text{meeting\_factor} \times S \times I$**2. Infected Population ($I$)**Infected people increase when susceptible people catch the disease, and decrease when infected people recover.$\frac{dI}{dt} = (\text{transmission\_rate} \times \text{meeting\_factor} \times S \times I) - (\text{recovery\_rate} \times I)$**3. Recovered Population ($R$)**Recovered people increase as infected people recover.$\frac{dR}{dt} = \text{recovery\_rate} \times I$

### Comprehension Check ✅1. What happens mathematically to $\frac{dS}{dt}$ if the number of Infected people ($I$) reaches 0?2. What is the sum of $\frac{dS}{dt} + \frac{dI}{dt} + \frac{dR}{dt}$ at any given moment, and what does this mean physically?<details><summary>💡 Answers</summary>1. It becomes 0. The disease is eradicated, so no one else can catch it. Susceptible population remains constant.2. The sum is exactly 0. This means the total population $N = S + I + R$ is strictly conserved. No one dies or is born in this simplified model; they just move between the three buckets.</details>

---## Phase 0 — Math Foundation Practice### 🎯 Your Turn: Translating Word Problems to CodeGiven:- Initial state: $S = 1000, I = 1, R = 0$.- Meeting factor: $0.01$- Transmission rate: $3\%$ ($0.03$)- Recovery rate: $2\%$ ($0.02$)Define the constants, then calculate $\frac{dS}{dt}$, $\frac{dI}{dt}$, and $\frac{dR}{dt}$ at $t=0$.

In [ ]:
S = 1000I = 1R = 0meeting_factor = 0.01transmission_rate = 0.03recovery_rate = 0.02# dS_dt = - transmission_rate * meeting_factor * S * I# dI_dt = (transmission_rate * meeting_factor * S * I) - (recovery_rate * I)# dR_dt = recovery_rate * I# print(f"dS/dt = {dS_dt}") # Should be -0.3# print(f"dI/dt = {dI_dt}") # Should be  0.28# print(f"dR/dt = {dR_dt}") # Should be  0.02

---## Phase 1 — Algorithm Construction### Step 1: Defining the Differential Equation FunctionsBring over your `MultiEulerEstimator` class from the previous notebook. Then, define the three derivative functions: `dS_dt(t, state)`, `dI_dt(t, state)`, `dR_dt(t, state)` where `state` is the tuple `(S, I, R)`.

In [ ]:
# PASTE MultiEulerEstimator HEREclass MultiEulerEstimator:    def __init__(self, funcs):        self.functions = funcs    def calc_derivatives_at_point(self, point):        t, state = point        return tuple(f(t, state) for f in self.functions)    def step(self, point, step_size):        t, state = point        slopes = self.calc_derivatives_at_point(point)        new_state = tuple(s + sl*step_size for s, sl in zip(state, slopes))        return (t + step_size, new_state)    def calc_estimated_points(self, point, step_size, target_time):        points = [point]        curr = point        while curr[0] < target_time:            curr = self.step(curr, step_size)            points.append(curr)        return points# DEFINE THE DIFFERENTIAL EQUATIONS HERE# def dS_dt(t, state):#     S, I, R = state#     return -0.03 * 0.01 * S * I# def dI_dt(t, state):#     # YOUR CODE HERE#     pass# def dR_dt(t, state):#     # YOUR CODE HERE#     pass

---## Phase 2 — Experimental Verification### Simulating the OutbreakInitialize your estimator. Start at `t = 0` with state `(1000, 1, 0)`. Use a step size of `0.1` and target time of `60` days. Plot $S$, $I$, and $R$ over time.

In [ ]:
# multi_est = MultiEulerEstimator([dS_dt, dI_dt, dR_dt])# pts = multi_est.calc_estimated_points((0.0, (1000.0, 1.0, 0.0)), step_size=0.1, target_time=60)# ts = [p[0] for p in pts]# s_vals = [p[1][0] for p in pts]# i_vals = [p[1][1] for p in pts]# r_vals = [p[1][2] for p in pts]# plt.plot(ts, s_vals, label="Susceptible (S)", color='blue')# plt.plot(ts, i_vals, label="Infected (I)", color='red')# plt.plot(ts, r_vals, label="Recovered (R)", color='green')# plt.legend(); plt.grid(True)# plt.title("SIR Epidemic Simulation")# plt.xlabel("Days"); plt.ylabel("Population")# plt.show()

### 🔍 ReflectionLook at your plot. At what day does the infection "peak"? Notice that the pandemic ends *not* because the virus magically vanishes, but because the Susceptible pool shrinks so low that the virus cannot find new hosts fast enough to outpace the recovery rate. This is the definition of **Herd Immunity**!

---## Phase 3 — Olympiad Extension### Challenge: Flattening the CurveWhat happens if we introduce social distancing? Modify the `meeting_factor` from `0.01` to `0.005`.Re-run the simulation. You will notice the red peak is much lower and delayed! But notice the tradeoff: the pandemic lasts much longer.*(Try modifying your functions and plotting the two I(t) curves on top of each other!)*

In [ ]:
# YOUR CODE HEREpass

### Chalkboard DefenseOur SIR model assumes that once you recover, you have permanent immunity (you join bucket $R$ and stay there forever). If the disease mutates and immunity only lasts 30 days, how would you change the differential equations to make this an **SIRS model**?<details><summary>💡 Sketch</summary>You would add a term that drains population from $R$ and puts it back into $S$ after a delay, or at a constant rate. $\frac{dS}{dt} = -(\dots) + \text{immunity\_loss\_rate} \times R$$\frac{dR}{dt} = \text{recovery\_rate} \times I - \text{immunity\_loss\_rate} \times R$This causes the disease to become *endemic*, creating a constant wave of infections that never permanently dies out.</details>---📚 **Next:** Algorithms 20 (Hodgkin-Huxley Model)